In [1]:
import os
import sys
import torch
from pathlib import Path

if "__file__" in globals():
    project_root = Path(__file__).resolve().parent.parent
else:
    project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

project_root = project_root.resolve()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

if Path.cwd() != project_root:
    os.chdir(project_root)

from torch.utils.data import TensorDataset, random_split
from core.config import load_config, print_config
from core.data.loader import setup_dataset, load_dataset
from core.data.dataset import create_dataloaders
from core.data.transforms import create_normalizer_from_data
from core.model.bert import BertForPretraining
from core.training.sampler import create_kde_sampler
from core.training.pretrainer import setup_training
from core.logger import print_data_summary, log_model_summary

In [2]:
print(f"PyTorch version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.set_float32_matmul_precision("high")
    if torch.cuda.is_bf16_supported():
        print("Bfloat16 is supported and will be used.")
print(f"Using device: {device}")
print(f"Working directory: {os.getcwd()}")

config = load_config("config", config_dir=".")
print_config(config, "Loaded BERT Configuration")

PyTorch version: 2.9.0+cu126
Bfloat16 is supported and will be used.
Using device: cuda
Working directory: /home/jessiez/projects/osu_corpora

--- Loaded BERT Configuration ---
data:
  max_seq_len: 2048
  val_split: 0.1
  max_samples_per_class:
    aim: 2500
    tech: 2500
  min_stars: 4.0
  max_stars: 12.0
model:
  d_model: 512
  n_heads: 8
  n_layers: 6
  dim_feedforward_mult: 4
  dropout: 0.1
  local_attention_window: 256
  cnn_kernel_size: 15
components:
  use_flash_attention: true
  compile_model: true
  compile_mode: default
pretraining:
  db_path: ./data/beatmap_dataset_test/
  batch_size: 8
  num_epochs: 8
  learning_rate: 0.0002
  min_lr: 1.0e-06
  cooldown_type: cosine
  weight_decay: 0.05
  warmup_ratio: 0.1
  stable_ratio: 0.1
  use_amp: true
  checkpoint_dir: ./checkpoints
  grad_clip_norm: 1.0
  gradient_accumulation_steps: 8
  difficulty_loss_weight: 1.0
  mlm_loss_weight: 1.0
  masking_ratio: 0.3
  mean_span_length: 4
  sampling:
    method: kde
    kde_bandwidth: 0.2
 

/home/jessiez/projects/osu_corpora/.venv/lib/python3.12/site-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


In [3]:
colab_url = 'https://drive.google.com/uc?id=14yvshmHQ069SCjKIa8uBak8aPcyccSqv'
DATASET_PATH = setup_dataset(config['pretraining']['db_path'], colab_url)

print(f"Using database: {DATASET_PATH}")

all_beatmaps_data, difficulty_attributes, loaded_ids = load_dataset(
    DATASET_PATH, 
    max_seq_len=config['data']['max_seq_len'],
    raw_beatmap_path=config['pretraining'].get('raw_beatmap_path', './data/raw')
)

print_data_summary(all_beatmaps_data)

Using database: ./data/beatmap_dataset_test/
Loading raw data from Parquet dataset...
Loading beatmap metadata...
Found metadata for 7376 beatmaps. Processing in chunks of 2000...


Processing Chunks:   0%|          | 0/4 [00:00<?, ?it/s]

Splitting sliders and spinners into head/end tokens...


Processing Chunks:  25%|██▌       | 1/4 [00:25<01:16, 25.60s/it]

Splitting sliders and spinners into head/end tokens...


...

Consolidating processed chunks...
Loaded raw feature vectors for 7367 beatmaps.
Calculating difficulty attributes (will use cache if available)...


Calculating Attributes: 100%|██████████| 10/10 [00:00<00:00, 116185.71it/s]


Running final data integrity check...


Validating Tensors: 100%|██████████| 7357/7357 [00:00<00:00, 9626.29it/s] 

Finished loading and processing all data.

--- Data Summary ---
Total beatmaps: 7357
Vector dimension: 18
Sequence length - Min: 20, Max: 2048, Avg: 827.4
--------------------


In [4]:
val_size = int(len(all_beatmaps_data) * config['data']['val_split'])
train_size = len(all_beatmaps_data) - val_size

temp_dataset = TensorDataset(torch.arange(len(all_beatmaps_data)))
train_split, val_split = random_split(temp_dataset, [train_size, val_size])

print(f"Data split: {len(train_split.indices)} training, {len(val_split.indices)} validation")

train_data_list = [all_beatmaps_data[i] for i in train_split.indices]
val_data_list = [all_beatmaps_data[i] for i in val_split.indices]

train_attributes = {key: val[train_split.indices] for key, val in difficulty_attributes.items()}
val_attributes = {key: val[val_split.indices] for key, val in difficulty_attributes.items()}

sampler = create_kde_sampler(
    train_attributes['stars'],
    bandwidth=config['pretraining']['sampling']['kde_bandwidth'],
    num_bins=config['pretraining']['sampling'].get('num_bins', 100),
)

normalizer = create_normalizer_from_data(train_data_list, train_attributes)
vector_stats = normalizer.get_vector_stats()

print(f"Vector normalization stats for {len(vector_stats)} fields")

Data split: 6622 training, 735 validation
Creating optimized KDE sampler with bandwidth=0.2, bins=200...
KDE sampling - Min weight: 0.1987, Max weight: 316.5082
Calculating normalization statistics...

                    NORMALIZATION STATISTICS

--- VECTOR STATISTICS:
------------------------------------------------------------
Field Name             Type         Param 1      Param 2     
------------------------------------------------------------
norm_x                 none         N/A          N/A         
norm_y                 none         N/A          N/A         
delta_x                mean/std     0.0002       110.0375    
delta_y                mean/std     0.0012       98.7434     
log_time_diff_ms       mean/std     4.8679       0.4983      
bpm                    mean/std     184.8940     37.7230     
notes_per_second       mean/std     8.1658       2.7903      
velocity               mean/std     0.9023       1.3844      
relative_angle         none         N/A          

In [5]:
train_dataloader, val_dataloader = create_dataloaders(
    train_data_list,
    val_data_list,
    train_attributes,
    val_attributes,
    normalizer,
    config, device, sampler
)

print(f"Created dataloaders with batch size: {config['pretraining']['batch_size']}")

sample_batch = next(iter(train_dataloader))
print(f"Sample batch shapes: vectors={sample_batch[0].shape}, mask={sample_batch[1].shape}")
print(f"Sample attributes keys: {list(sample_batch[2].keys())}")

Created dataloaders with batch size: 8
Sample batch shapes: vectors=torch.Size([8, 2048, 18]), mask=torch.Size([8, 2048])
Sample attributes keys: ['stars', 'aim', 'speed', 'slider_factor', 'hp', 'cs', 'od', 'ar', 'slider_multiplier']


In [6]:
model = BertForPretraining.from_config(config, device)
log_model_summary(model)

print("\nRunning a test forward pass with mixed precision (autocast)...")
use_amp_for_test = device.type == "cuda"
with torch.no_grad():
    with torch.amp.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=use_amp_for_test):
        sample_vectors, sample_mask, sample_attrs, sample_cu_seqlens = sample_batch
        
        sample_vectors = sample_vectors.to(device)
        sample_mask = sample_mask.to(device)

        predictions, targets, _ = model(sample_vectors, sample_mask)

        print(f"Prediction output keys: {list(predictions.keys())}")
        print(f"MLM prediction keys: {list(predictions['mlm'].keys())}")
        print(f"Difficulty prediction keys: {list(predictions['difficulty'].keys())}")

print("\nBERT model created and tested successfully!")

Compiling BERT pre-training model with torch.compile...

--- BERT Encoder Information ---
Total Parameters: 33.27M
Model Dimension: 512
Number of Heads: 8
Number of Layers: 6
Flash Attention: True
------------------------------

--- Pre-training Head Information ---
Tasks: Masked Modeling, Difficulty Attribute Prediction
Masking Ratio: 0.3
Model Compiled: True
------------------------------

Running a test forward pass with mixed precision (autocast)...


W1224 18:17:10.144000 202515 .venv/lib/python3.12/site-packages/torch/_inductor/utils.py:1558] [11/0_1] Not enough SMs to use max_autotune_gemm mode


Prediction output keys: ['mlm', 'difficulty']
MLM prediction keys: ['continuous', 'categorical']
Difficulty prediction keys: ['stars', 'aim', 'speed', 'slider_factor', 'hp', 'cs', 'od', 'ar', 'slider_multiplier']

BERT model created and tested successfully!


In [7]:
trainer, checkpoint_manager = setup_training(
    model, train_dataloader, val_dataloader, config, device, normalizer
)

start_epoch = 0
if checkpoint_manager.checkpoint_exists():
    try:
        loaded_epoch, metrics = checkpoint_manager.load_checkpoint(
            model, trainer.optimizer, trainer.scheduler, trainer.scaler, device=device
        )
        start_epoch = loaded_epoch + 1 
        print(f"Loaded checkpoint from epoch {loaded_epoch}, resuming from epoch {start_epoch}")
        print(f"Previous metrics: {metrics}")
    except Exception as e:
        print(f"Could not load checkpoint: {e}")
        print("Starting pretraining from scratch")

print(f"Pretraining setup complete. Starting from epoch {start_epoch}")
print(f"Total epochs: {config['pretraining']['num_epochs']}")

Scheduler: WSD with 83 warmup, 83 stable, 666 decay steps.
Cooldown type: cosine, Min LR Ratio: 0.0050
PreTrainer initialized - AMP: True, Device: cuda, Grad Accum: 8
Effective batch size: 64
Pretraining setup complete. Starting from epoch 0
Total epochs: 8


In [8]:
print("\nStarting BERT pretraining...")
print(f"BERT Model: {config['model']['n_layers']} layers, {config['model']['d_model']} dimensions")

print(f"Pretraining samples: {len(train_data_list)} base maps")
print(f"Validation samples: {len(val_data_list)} base maps")

metrics_tracker = trainer.train(start_epoch)

print("\nBERT training completed!")


Starting BERT pretraining...
BERT Model: 6 layers, 512 dimensions
Pretraining samples: 6622 base maps
Validation samples: 735 base maps

--- Starting Training ---
Epochs: 1 to 8
Batch Size: 8
Learning Rate: 0.0002
------------------------------


Epoch 1 [Train]:   0%|          | 0/104 [00:00<?, ?it/s]

Epoch 1 [Validate]:   0%|          | 0/92 [00:00<?, ?it/s]

Epoch 1/8 | Train Loss: 63.7417 | Val Loss: 10.2630 | LR: 2.00e-04 | Time: 144.29s
------------------------------
Validation Metrics
Difficulty MAE:
  aim            : 0.3625
  ar             : 0.3251
  cs             : 0.3831
  hp             : 0.9308
  od             : 0.4783
  slider_factor  : 0.0189
  slider_multiplier: 0.3070
  speed          : 0.3275
  stars          : 0.4918
Continuous Features:
  norm_x         : MAE 0.4339
  norm_y         : MAE 0.4457
  delta_x        : MAE 0.7280
  delta_y        : MAE 0.7295
  log_time_diff_ms: MAE 0.5765
  bpm            : MAE 0.8487
  notes_per_second: MAE 0.5323
  velocity       : MAE 0.3396
  relative_angle : MAE 0.8387
  rhythm_change  : MAE 0.0695
  log_slider_pixel_length: MAE 0.9398
  slider_repeats : MAE 0.2031
  slider_tortuosity: MAE 0.2667
Categorical Features:
  object_type    : Acc 49.97%, Prec 0.3166, Rec 0.2262
  is_new_combo   : Acc 80.85%, Prec 0.4042, Rec 0.5000
  beat_in_measure: Acc 39.25%, Prec 0.2835, Rec 0.1261
  tim

Epoch 2 [Train]:   0%|          | 0/104 [00:00<?, ?it/s]

...

Epoch 7 [Validate]:   0%|          | 0/92 [00:00<?, ?it/s]

Epoch 7/8 | Train Loss: 4.9960 | Val Loss: 5.1882 | LR: 1.27e-05 | Time: 114.07s
------------------------------
Validation Metrics
Difficulty MAE:
  aim            : 0.1485
  ar             : 0.2037
  cs             : 0.4042
  hp             : 0.6974
  od             : 0.3835
  slider_factor  : 0.0215
  slider_multiplier: 0.2722
  speed          : 0.1095
  stars          : 0.2402
Continuous Features:
  norm_x         : MAE 0.2095
  norm_y         : MAE 0.2362
  delta_x        : MAE 0.5365
  delta_y        : MAE 0.5378
  log_time_diff_ms: MAE 0.4079
  bpm            : MAE 0.0608
  notes_per_second: MAE 0.1825
  velocity       : MAE 0.2556
  relative_angle : MAE 0.6839
  rhythm_change  : MAE 0.0389
  log_slider_pixel_length: MAE 0.7139
  slider_repeats : MAE 0.1484
  slider_tortuosity: MAE 0.1861
Categorical Features:
  object_type    : Acc 77.29%, Prec 0.4650, Rec 0.4493
  is_new_combo   : Acc 84.48%, Prec 0.7839, Rec 0.6417
  beat_in_measure: Acc 79.33%, Prec 0.3920, Rec 0.3907
  time_

Epoch 8 [Train]:   0%|          | 0/104 [00:00<?, ?it/s]

Epoch 8 [Validate]:   0%|          | 0/92 [00:00<?, ?it/s]

Epoch 8/8 | Train Loss: 4.8243 | Val Loss: 5.1022 | LR: 1.00e-06 | Time: 117.81s
------------------------------
Validation Metrics
Difficulty MAE:
  aim            : 0.1428
  ar             : 0.2069
  cs             : 0.3728
  hp             : 0.6919
  od             : 0.3777
  slider_factor  : 0.0221
  slider_multiplier: 0.2719
  speed          : 0.1123
  stars          : 0.2308
Continuous Features:
  norm_x         : MAE 0.2079
  norm_y         : MAE 0.2350
  delta_x        : MAE 0.5331
  delta_y        : MAE 0.5337
  log_time_diff_ms: MAE 0.4003
  bpm            : MAE 0.0606
  notes_per_second: MAE 0.1778
  velocity       : MAE 0.2536
  relative_angle : MAE 0.6831
  rhythm_change  : MAE 0.0354
  log_slider_pixel_length: MAE 0.7118
  slider_repeats : MAE 0.1546
  slider_tortuosity: MAE 0.1856
Categorical Features:
  object_type    : Acc 77.59%, Prec 0.4664, Rec 0.4520
  is_new_combo   : Acc 84.45%, Prec 0.7846, Rec 0.6416
  beat_in_measure: Acc 79.53%, Prec 0.3945, Rec 0.3905
  time_